In [ ]:
# 必要パッケージ
!pip install datasets==3.6.0
!pip install haystack-ai elasticsearch-haystack
!pip install evaluate

In [ ]:
## データセット用意
from datasets import load_dataset
subjqa = load_dataset("subjqa", name="electronics", trust_remote_code=True)
import pandas as pd
dfs = {split: dset.to_pandas() for split, dset in subjqa.flatten().items()}

In [ ]:
## haystack-ai v2 用のドキュメントストア構築（Elasticsearchなしで実行可）
from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore

train_df = dfs["train"].copy()
context_col = "context" if "context" in train_df.columns else "review.body"
item_col_candidates = ["title", "item_id", "asin", "product_id"]
item_col = next((c for c in item_col_candidates if c in train_df.columns), None)
question_id_col = "question_id" if "question_id" in train_df.columns else ("id" if "id" in train_df.columns else None)

doc_records = train_df.drop_duplicates(subset=[context_col]).reset_index(drop=True)
documents = []

for idx, row in doc_records.iterrows():
    meta = {"split": "train"}
    if item_col is not None:
        meta["item_id"] = str(row[item_col])
    if question_id_col is not None:
        meta["question_id"] = str(row[question_id_col])
    documents.append(Document(content=str(row[context_col]), meta=meta))

document_store = InMemoryDocumentStore()
document_store.write_documents(documents)
print(f"Indexed {len(documents)} documents into InMemoryDocumentStore")

In [ ]:
### BM25 Retriever のインスタンス化
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever

retriever = InMemoryBM25Retriever(document_store=document_store)

In [ ]:
# PromptBuilder と Generator のインスタンス化
from haystack.components.builders import PromptBuilder
from haystack.components.generators import HuggingFaceLocalGenerator

template = """
You are an extractive QA assistant.
Answer using only the given context.
If the answer is not in context, return exactly: not found

Context:
{% for doc in documents %}
- {{ doc.content }}
{% endfor %}

Question: {{ question }}
Answer:
"""

prompt_builder = PromptBuilder(template=template)
generator = HuggingFaceLocalGenerator(
    model="google/flan-t5-small",
    task="text2text-generation",
    generation_kwargs={
        "do_sample": False,
        "num_beams": 5,
    },
 )

generator.warm_up()

In [ ]:
# RAG Pipeline のインスタンス化
from haystack import Pipeline

rag_pipeline = Pipeline()
rag_pipeline.add_component("retriever", retriever)
rag_pipeline.add_component("prompt_builder", prompt_builder)
rag_pipeline.add_component("generator", generator)

rag_pipeline.connect("retriever.documents", "prompt_builder.documents")
rag_pipeline.connect("prompt_builder.prompt", "generator.prompt")

In [ ]:
# 回答と根拠文脈を表示
def _short_extractive_text(text, max_words=5):
    text = (text or "").strip().replace("\n", " ")
    if not text:
        return text
    words = text.split()
    return " ".join(words[:max_words])


def _build_item_filter(item_id=None):
    if item_id is None:
        return None
    return {"field": "meta.item_id", "operator": "==", "value": str(item_id)}


def generate_answers(query, item_id=None, top_k_retriever=5, top_k_context_preview=3):
    retriever_input = {"query": query, "top_k": top_k_retriever}
    item_filter = _build_item_filter(item_id)
    if item_filter is not None:
        retriever_input["filters"] = item_filter

    result = rag_pipeline.run(
        data={
            "retriever": retriever_input,
            "prompt_builder": {"question": query},
        },
        include_outputs_from={"retriever", "generator"},
    )

    replies = result.get("generator", {}).get("replies", [])
    docs = result.get("retriever", {}).get("documents", [])

    print(f"Question: {query}")
    print(f"Item filter: {item_id if item_id is not None else 'None (global search)'}\n")

    if replies:
        answer = _short_extractive_text(str(replies[0]), max_words=5)
        print(f"Answer: {answer}\n")
    else:
        print("Answer: (生成結果なし)\n")

    print("Retrieved contexts:")
    if not docs:
        print("(取得ドキュメントなし)\n")
        return

    for i, doc in enumerate(docs[:top_k_context_preview], start=1):
        score = f"{doc.score:.3f}" if doc.score is not None else "N/A"
        doc_item_id = doc.meta.get("item_id", "N/A")
        snippet = doc.content.replace("\n", " ")[:180]
        print(f"{i}. score={score}, item_id={doc_item_id}")
        print(f"   {snippet}...\n")

In [ ]:
# 質問（item_id を指定して同一商品の文脈に限定）
generate_answers("What is the main drawback?", item_id="B0074BW614")